In [15]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from toolsos.huisstijl.colors import get_os_colors
from graphs import simple_barh, simple_line, multiple_line
import pickle
from itertools import cycle

In [16]:
zo_onbewerkt = pd.read_excel('../../04 tabellen/01 tabellen zuidoost/tabel alle data long Masterplan Zuidoost 2026_05.xlsx')

In [17]:
len(zo_onbewerkt.measure.unique())

130

In [18]:
def bewerk_dataframe(df_onbewerkt):
    df = df_onbewerkt.loc[
        (df_onbewerkt["spatial_type"].isin(["stadsdelen", "gemeente"]))  &
        (df_onbewerkt['tweedeling_def'] == 'totaal')
    ].copy()

    df["temporal_date"] = pd.to_datetime(df["temporal_date"], format="%Y")
    df["temporal_date"] = df["temporal_date"].dt.year.astype(str)

    #df = df.sort_values(["spatial_name", "measure", "temporal_date"])

    return df

zo = bewerk_dataframe(zo_onbewerkt)

In [ ]:
indicatoren = zo['measure'].unique()
kernindicatoren = zo.loc[zo['kernindicator_zo']==1]['measure'].unique()
indicatoren = [indicator for indicator in indicatoren if indicator not in kernindicatoren]

kleuren = get_os_colors(type='discreet', kleur='discreet (1-9)', aantal='9')  # Lijst met kleuren
kleur_mapping = {kernindicator: kleur for kernindicator, kleur in zip(kernindicatoren, cycle(kleuren))}


In [22]:
zo.loc[zo['measure']=='vkparkeren_r']

,thema_zuidoost_label,kernindicator_zo,indicator_sd,measure,spatial_code,spatial_name,spatial_type,tweedeling_def,temporal_date,value,thema_zuidoost_code
2704,SD3 Openbare ruimte en mobiliteit,1.0,oordeel aanbod parkeervoorzieningen (1-10),vkparkeren_r,T,Zuidoost,stadsdelen,totaal,2019,6.4,SD3
2705,SD3 Openbare ruimte en mobiliteit,1.0,oordeel aanbod parkeervoorzieningen (1-10),vkparkeren_r,0363,Amsterdam,gemeente,totaal,2019,6.3,SD3
9090,SD3 Openbare ruimte en mobiliteit,1.0,oordeel aanbod parkeervoorzieningen (1-10),vkparkeren_r,T,Zuidoost,stadsdelen,totaal,2021,6.6,SD3
9091,SD3 Openbare ruimte en mobiliteit,1.0,oordeel aanbod parkeervoorzieningen (1-10),vkparkeren_r,0363,Amsterdam,gemeente,totaal,2021,6.5,SD3
16166,SD3 Openbare ruimte en mobiliteit,1.0,oordeel aanbod parkeervoorzieningen (1-10),vkparkeren_r,T,Zuidoost,stadsdelen,totaal,2023,6.3,SD3
16167,SD3 Openbare ruimte en mobiliteit,1.0,oordeel aanbod parkeervoorzieningen (1-10),vkparkeren_r,0363,Amsterdam,gemeente,totaal,2023,6.4,SD3


#### Bars

In [95]:
for indicator in kernindicatoren:
    df = zo[(zo['measure']==indicator) & (zo['spatial_type']=='stadsdelen')]
    if df['value'].notna().any() and '_r' in indicator:
        
        kleur = kleur_mapping.get(indicator, '#004699') # Zoek de kleur op in de mapping

        simple_barh(
            df=df,
            x_col='temporal_date',
            y_col='value',
            y_min=0,
            y_max=10,
            color_func=kleur,
            output_path=f'../../05 reports/Zuidoost/andere indicatoren/{indicator}.png' 
        )

#### Multiple Line Ams en SD

In [94]:
def multiple_line_perc_figuur(indicator_list, output_folder):
    for indicator in indicator_list:
        
        df = zo[zo['measure'] == indicator]

        if df['value'].notna().any():

            kleur = kleur_mapping.get(indicator, '#004699')  # Zoek de kleur op in de mapping

            # Controleer of '_p' of '_r' in de indicatornaam staat en pas y-as daarop aan
            if '_p' in indicator:
                y_min = 0
                y_max = 100 if df['value'].max() > 20 else 30
                is_percentage = True
            elif '_r' in indicator:
                y_min = 1
                y_max = 10
                is_percentage = False
            else:
                # Gebruik standaardwaarden als geen '_p' of '_r' aanwezig is
                y_min = None
                y_max = None
                is_percentage = False 
                                    
            multiple_line(
                df=df,
                x_col='temporal_date',
                y_col='value',
                main_area='Amsterdam', 
                y_min= y_min,
                y_max= y_max,
                color_func=kleur,  # Gebruik de kleur uit de mapping
                is_percentage=is_percentage,
                output_path=f'../../05 reports/Zuidoost/{output_folder}/{indicator}.png'
                )

# Verwerk indicatoren
multiple_line_perc_figuur(indicatoren, 'andere indicatoren')

# Verwerk kernindicatoren
multiple_line_perc_figuur(kernindicatoren, 'kernindicatoren')

In [87]:
zo.to_excel('../../05 reports/Zuidoost/data_indicatoren.xlsx')